In [1]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
from enum import Enum
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

# Agent Types
class AgentType(Enum):
    PASSIVE = 1
    NORMAL = 2
    AGGRESSIVE = 3

TYPE_COEFFICIENTS = {
    AgentType.PASSIVE: 0.3,
    AgentType.NORMAL: 0.7,
    AgentType.AGGRESSIVE: 1.0
}

# Random Event Types
class EventType(Enum):
    NONE = 0
    ANNOUNCEMENT = 1    # Urgency spike across all agents
    DISTRACTION = 2     # Agent temporarily frozen
    LATE_ARRIVAL = 3    # High urgency agent joins rear
    ALTERCATION = 4     # Local blockage
    DISORIENTED = 5       # Agent moves randomly

print("Imports successful!")
print(f"Mesa version: {mesa.__version__}")
print("Random event system: ENABLED")

Imports successful!
Mesa version: 3.5.1
Random event system: ENABLED


In [2]:
class QueueAgent(mesa.Agent):
    def __init__(self, model, agent_type):
        super().__init__(model)
        self.agent_type = agent_type
        self.type_coeff = TYPE_COEFFICIENTS[agent_type]
        
        # Individual characteristics
        self.urgency = np.random.uniform(0.3, 1.0)
        self.social_inhibition = np.random.uniform(0.2, 0.8)
        self.exited = False
        
        # Latency tracking
        self.entry_time = None
        self.exit_time = None
        self.latency = None
        self.steps_waiting = 0
        self.steps_moving = 0
        
        # Human unpredictability
        self.frozen = False          # distraction event
        self.frozen_countdown = 0    # steps remaining frozen
        self.urgency_boost = 1.0     # announcement multiplier
        self.confused = False        # DISORIENTED event
        self.confused_countdown = 0  # steps remaining confused
        
    @property
    def behavior_score(self):
        if self.frozen:
            return 0.0
            
        base_score = (self.type_coeff * 
                     self.urgency * 
                     self.urgency_boost /
                     self.social_inhibition)
        
        # Density modifier
        if self.pos:
            neighbors = self.model.grid.get_neighbors(
                self.pos,
                moore=True,
                include_center=False,
                radius=2
            )
            density_factor = 1.0 + (len(neighbors) * 0.05)
        else:
            density_factor = 1.0
            
        return base_score * density_factor
    
    def handle_events(self):
        """Process any active random events"""
        # Count down frozen state
        if self.frozen:
            self.frozen_countdown -= 1
            if self.frozen_countdown <= 0:
                self.frozen = False
                
        # Count down DISORIENTED
        if self.confused:
            self.confused_countdown -= 1
            if self.confused_countdown <= 0:
                self.confused = False
        
        # Decay urgency boost gradually
        if self.urgency_boost > 1.0:
            self.urgency_boost = max(
                1.0, 
                self.urgency_boost - 0.05
            )
    
    def step(self):
        if self.exited:
            return
            
        current_pos = self.pos
        if current_pos is None:
            return
        
        # Record entry time
        if self.entry_time is None:
            self.entry_time = self.model.steps
        
        # Process events
        self.handle_events()
        
        # If frozen do nothing this step
        if self.frozen:
            self.steps_waiting += 1
            return
            
        x, y = current_pos
        moved = False
        
        # Confused agents move randomly
        if self.confused:
            new_x = x + np.random.randint(-1, 2)
            new_x = max(0, min(new_x, self.model.width - 1))
            new_y = y + np.random.randint(-1, 1)
            new_y = max(0, min(new_y, self.model.height - 1))
            self.model.grid.move_agent(self, (new_x, new_y))
            self.steps_waiting += 1
            return
        
        move_prob = min(self.behavior_score, 1.0)
        
        if np.random.random() < move_prob:
            new_y = y - 1
            
            if new_y < 0:
                self.model.grid.remove_agent(self)
                self.exited = True
                self.exit_time = self.model.steps
                if self.entry_time is not None:
                    self.latency = (self.exit_time - 
                                   self.entry_time)
                self.model.exited_count += 1
                self.model.exit_times.append(self.model.steps)
                self.model.record_latency(self)
                return
            
            new_pos = (x, new_y)
            cell_contents = self.model.grid.get_cell_list_contents(
                [new_pos]
            )
            
            max_occupancy = (
                3 if self.agent_type == AgentType.AGGRESSIVE
                else 2 if self.agent_type == AgentType.NORMAL
                else 1
            )
            
            if len(cell_contents) < max_occupancy:
                self.model.grid.move_agent(self, new_pos)
                self.steps_moving += 1
                moved = True
        
        if not moved:
            self.steps_waiting += 1

print("Enhanced agent class defined!")
print("Random event responses: ENABLED")
print("Tracking: frozen, confused, urgency_boost")

Enhanced agent class defined!
Random event responses: ENABLED
Tracking: frozen, confused, urgency_boost


In [3]:
class BunchQueueModel(mesa.Model):
    def __init__(self, n_agents=100, width=20, height=30,
                 pct_aggressive=0.15, pct_normal=0.25,
                 enable_events=True):
        super().__init__()
        
        self.width = width
        self.height = height
        self.steps = 0
        self.exited_count = 0
        self.exit_times = []
        self.throughput_history = []
        self.enable_events = enable_events
        self.total_agents = n_agents
        self.event_log = []
        self.latencies = {
            AgentType.PASSIVE: [],
            AgentType.NORMAL: [],
            AgentType.AGGRESSIVE: []
        }
        self.all_latencies = []
        self.grid_history = []
        
        self.grid = mesa.space.MultiGrid(
            width, height, torus=False
        )
        
        n_aggressive = int(n_agents * pct_aggressive)
        n_normal = int(n_agents * pct_normal)
        n_passive = n_agents - n_aggressive - n_normal
        
        agent_types = (
            [AgentType.AGGRESSIVE] * n_aggressive +
            [AgentType.NORMAL] * n_normal +
            [AgentType.PASSIVE] * n_passive
        )
        np.random.shuffle(agent_types)
        
        for agent_type in agent_types:
            agent = QueueAgent(self, agent_type)
            x = np.random.randint(0, width)
            y = np.random.randint(height // 2, height)
            self.grid.place_agent(agent, (x, y))
    
    def record_latency(self, agent):
        if agent.latency is not None:
            self.latencies[agent.agent_type].append(
                agent.latency
            )
            self.all_latencies.append(agent.latency)
    
    def trigger_random_event(self):
        rand = np.random.random()
        active_agents = [
            a for a in self.agents
            if not a.exited and a.pos is not None
        ]
        
        # Guard against too few agents
        if len(active_agents) < 2:
            return
        
        n = len(active_agents)
        
        # Announcement - urgency spike (5% chance)
        if rand < 0.05:
            affected = np.random.randint(
                1, max(2, min(20, n))
            )
            targets = np.random.choice(
                active_agents, affected, replace=False
            )
            for agent in targets:
                agent.urgency_boost = np.random.uniform(
                    1.5, 2.5
                )
            self.event_log.append({
                'step': self.steps,
                'type': EventType.ANNOUNCEMENT,
                'affected': affected
            })
        
        # Distraction - agent frozen (8% chance)
        elif rand < 0.13:
            target = np.random.choice(active_agents)
            target.frozen = True
            target.frozen_countdown = np.random.randint(3, 8)
            self.event_log.append({
                'step': self.steps,
                'type': EventType.DISTRACTION,
                'affected': 1
            })
        
        # Late arrival (3% chance)
        elif rand < 0.16:
            agent = QueueAgent(self, AgentType.AGGRESSIVE)
            agent.urgency = 0.95
            agent.urgency_boost = 2.0
            x = np.random.randint(0, self.width)
            y = np.random.randint(
                self.height - 5, self.height
            )
            self.grid.place_agent(agent, (x, y))
            self.total_agents += 1
            self.event_log.append({
                'step': self.steps,
                'type': EventType.LATE_ARRIVAL,
                'affected': 1
            })
        
        # DISORIENTED (4% chance)
        elif rand < 0.20:
            target = np.random.choice(active_agents)
            target.confused = True
            target.confused_countdown = np.random.randint(2, 5)
            self.event_log.append({
                'step': self.steps,
                'type': EventType.DISORIENTED,
                'affected': 1
            })

    # ALTERCATION - future enhancement
        # Two aggressive agents contest same space
        # creating local pressure wave and temporary blockage
        # Hypothesis: explains thrashing threshold at 30% aggression
        # elif rand < 0.24:
        #     pass  # TODO: implement in future version
        
    def capture_grid_state(self):
        state = np.zeros((self.height, self.width))
        for agent in self.agents:
            if agent.pos and not agent.exited:
                x, y = agent.pos
                if agent.frozen:
                    state[y][x] = 4
                elif agent.confused:
                    state[y][x] = 5
                elif agent.agent_type == AgentType.AGGRESSIVE:
                    state[y][x] = 3
                elif agent.agent_type == AgentType.NORMAL:
                    state[y][x] = 2
                else:
                    state[y][x] = 1
        return state
    
    def step(self):
        self.steps += 1
        
        if self.enable_events:
            self.trigger_random_event()
        
        self.agents.shuffle_do("step")
        self.throughput_history.append(self.exited_count)
        
        if self.steps % 2 == 0:
            self.grid_history.append(
                self.capture_grid_state()
            )
    
    def run(self, max_steps=300):
        for _ in range(max_steps):
            self.step()
            if self.exited_count >= self.total_agents:
                break
        return self.exited_count, self.exit_times

print("Enhanced model class defined!")
print("Random events: ENABLED")
print("Guard against low agents: FIXED")
print("Animation ready: YES")

Enhanced model class defined!
Random events: ENABLED
Guard against low agents: FIXED
Animation ready: YES


In [4]:
# Run simulation with events enabled
print("Running V5 simulation with random events...")
print("="*50)

model = BunchQueueModel(
    n_agents=100,
    width=20,
    height=30,
    pct_aggressive=0.15,
    pct_normal=0.25,
    enable_events=True
)

exited, exit_times = model.run(max_steps=300)

print(f"Agents exited: {exited}")
print(f"Steps taken: {model.steps}")
print(f"Grid states captured: {len(model.grid_history)}")
print(f"\nRandom Events That Occurred:")
print("-"*50)

# Summarize events
event_counts = {}
for event in model.event_log:
    etype = event['type'].name
    event_counts[etype] = event_counts.get(etype, 0) + 1

for etype, count in event_counts.items():
    print(f"  {etype:<20} {count} times")

print(f"\nTotal events: {len(model.event_log)}")
print("="*50)

Running V5 simulation with random events...
Agents exited: 104
Steps taken: 322
Grid states captured: 161

Random Events That Occurred:
--------------------------------------------------
  ANNOUNCEMENT         10 times
  DISORIENTED          8 times
  DISTRACTION          11 times
  LATE_ARRIVAL         4 times

Total events: 33


In [5]:
# Run simulation with events enabled
print("Running V5 simulation with random events...")
print("="*50)

model = BunchQueueModel(
    n_agents=100,
    width=20,
    height=30,
    pct_aggressive=0.15,
    pct_normal=0.25,
    enable_events=True
)

exited, exit_times = model.run(max_steps=300)

print(f"Agents exited: {exited}")
print(f"Steps taken: {model.steps}")
print(f"Grid states captured: {len(model.grid_history)}")
print(f"\nRandom Events That Occurred:")
print("-"*50)

event_counts = {}
for event in model.event_log:
    etype = event['type'].name
    event_counts[etype] = event_counts.get(etype, 0) + 1

for etype, count in event_counts.items():
    print(f"  {etype:<20} {count} times")

print(f"\nTotal events: {len(model.event_log)}")
print("="*50)

Running V5 simulation with random events...
Agents exited: 104
Steps taken: 356
Grid states captured: 178

Random Events That Occurred:
--------------------------------------------------
  ANNOUNCEMENT         9 times
  DISORIENTED          8 times
  DISTRACTION          10 times
  LATE_ARRIVAL         4 times

Total events: 31


In [6]:
# Build the animation
print("Building animation...")

# Color map
# 0=empty, 1=passive, 2=normal, 3=aggressive
# 4=frozen, 5=confused
colors = ['white', 'steelblue', 'orange', 
          'red', 'lightgray', 'purple']
cmap = ListedColormap(colors)

fig, (ax1, ax2) = plt.subplots(
    1, 2, 
    figsize=(14, 8),
    gridspec_kw={'width_ratios': [2, 1]}
)

# Initial frame
im = ax1.imshow(
    model.grid_history[0],
    cmap=cmap,
    vmin=0, vmax=5,
    interpolation='nearest',
    aspect='auto'
)

ax1.set_title('Bunch Queue Simulation\nStep 0', 
              fontsize=12, fontweight='bold')
ax1.set_xlabel('Queue Width')
ax1.set_ylabel('Queue Length')
ax1.axhline(y=14, color='green', 
            linestyle='--', alpha=0.5,
            label='Entry threshold')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Passive'),
    Patch(facecolor='orange', label='Normal'),
    Patch(facecolor='red', label='Aggressive'),
    Patch(facecolor='lightgray', label='Frozen'),
    Patch(facecolor='purple', label='Confused'),
]
ax1.legend(handles=legend_elements, 
           loc='upper right', fontsize=8)

# Throughput chart
throughput_line, = ax2.plot(
    [], [], 'b-', linewidth=2
)
ax2.set_xlim(0, len(model.throughput_history))
ax2.set_ylim(0, model.total_agents + 5)
ax2.set_title('Throughput Over Time', 
              fontsize=12, fontweight='bold')
ax2.set_xlabel('Steps')
ax2.set_ylabel('Agents Exited')
ax2.grid(True, alpha=0.3)

# Event markers
event_steps = [e['step'] for e in model.event_log]
event_types = [e['type'] for e in model.event_log]
event_colors = {
    EventType.ANNOUNCEMENT: 'gold',
    EventType.DISTRACTION: 'gray',
    EventType.LATE_ARRIVAL: 'red',
    EventType.DISORIENTED: 'purple'
}

step_counter = ax2.text(
    0.02, 0.95, 'Step: 0',
    transform=ax2.transAxes,
    fontsize=9, verticalalignment='top'
)
exited_counter = ax2.text(
    0.02, 0.88, 'Exited: 0',
    transform=ax2.transAxes,
    fontsize=9, verticalalignment='top'
)
def animate(frame):
    # Update grid
    im.set_array(model.grid_history[frame])
    ax1.set_title(
        f'Bunch Queue Simulation\nStep {frame * 2}',
        fontsize=12, fontweight='bold'
    )
    
    # Update throughput
    step = frame * 2
    throughput_data = model.throughput_history[:step]
    throughput_line.set_data(
        range(len(throughput_data)),
        throughput_data
    )
    
    # Update counters - safe index
    if step > 0 and step <= len(model.throughput_history):
        exited = model.throughput_history[
            min(step - 1, 
                len(model.throughput_history) - 1)
        ]
    else:
        exited = 0
        
    step_counter.set_text(f'Step: {step}')
    exited_counter.set_text(f'Exited: {exited}')
    
    return [im, throughput_line,
            step_counter, exited_counter]

# Create animation
anim = animation.FuncAnimation(
    fig,
    animate,
    frames=len(model.grid_history),
    interval=100,
    blit=True
)

# Save as gif
print("Saving animation as GIF...")
anim.save(
    'bunch_queue_animation.gif',
    writer='pillow',
    fps=10,
    dpi=80
)

plt.tight_layout()
plt.close()
print("Animation saved as bunch_queue_animation.gif!")
print("File location: ~/bunch_queue_animation.gif")


Building animation...
Saving animation as GIF...
Animation saved as bunch_queue_animation.gif!
File location: ~/bunch_queue_animation.gif


In [7]:
# Statistical comparison - events vs no events
import numpy as np

n_runs = 20  # runs per scenario
results = {
    'with_events': {
        'exited': [],
        'steps': [],
        'total_events': [],
        'latencies': []
    },
    'without_events': {
        'exited': [],
        'steps': [],
        'total_events': [],
        'latencies': []
    }
}

print("STATISTICAL COMPARISON")
print("Events vs No Events — 20 runs each")
print("="*50)
print("Running simulations...")

for i in range(n_runs):
    # With events
    m1 = BunchQueueModel(
        n_agents=100,
        width=20,
        height=30,
        pct_aggressive=0.15,
        pct_normal=0.25,
        enable_events=True
    )
    m1.run(max_steps=500)
    results['with_events']['exited'].append(
        m1.exited_count
    )
    results['with_events']['steps'].append(m1.steps)
    results['with_events']['total_events'].append(
        len(m1.event_log)
    )
    if m1.all_latencies:
        results['with_events']['latencies'].extend(
            m1.all_latencies
        )
    
    # Without events
    m2 = BunchQueueModel(
        n_agents=100,
        width=20,
        height=30,
        pct_aggressive=0.15,
        pct_normal=0.25,
        enable_events=False
    )
    m2.run(max_steps=500)
    results['without_events']['exited'].append(
        m2.exited_count
    )
    results['without_events']['steps'].append(m2.steps)
    results['without_events']['total_events'].append(0)
    if m2.all_latencies:
        results['without_events']['latencies'].extend(
            m2.all_latencies
        )
    
    print(f"  Run {i+1:2d}/20 complete")

print("\nRESULTS SUMMARY")
print("="*50)

for scenario, data in results.items():
    label = "With Events   " if scenario == 'with_events' else "Without Events"
    avg_exited = np.mean(data['exited'])
    std_exited = np.std(data['exited'])
    avg_steps = np.mean(data['steps'])
    std_steps = np.std(data['steps'])
    avg_latency = np.mean(data['latencies'])
    p95_latency = np.percentile(data['latencies'], 95)
    
    if scenario == 'with_events':
        avg_events = np.mean(data['total_events'])
    else:
        avg_events = 0
    
    print(f"\n{label}")
    print(f"  Avg exited:      {avg_exited:.1f} ± {std_exited:.1f}")
    print(f"  Avg steps:       {avg_steps:.1f} ± {std_steps:.1f}")
    print(f"  Avg events/run:  {avg_events:.1f}")
    print(f"  Avg latency:     {avg_latency:.1f} steps")
    print(f"  P95 latency:     {p95_latency:.1f} steps")

# Impact calculation
avg_steps_with = np.mean(results['with_events']['steps'])
avg_steps_without = np.mean(results['without_events']['steps'])
impact = ((avg_steps_with - avg_steps_without) / 
          avg_steps_without * 100)

print("\n" + "="*50)
print(f"HUMAN UNPREDICTABILITY IMPACT")
print("="*50)
print(f"Steps increase due to events: {impact:.1f}%")
print(f"This represents the cost of human")
print(f"unpredictability on queue throughput")
print("="*50)

STATISTICAL COMPARISON
Events vs No Events — 20 runs each
Running simulations...
  Run  1/20 complete
  Run  2/20 complete
  Run  3/20 complete
  Run  4/20 complete
  Run  5/20 complete
  Run  6/20 complete
  Run  7/20 complete
  Run  8/20 complete
  Run  9/20 complete
  Run 10/20 complete
  Run 11/20 complete
  Run 12/20 complete
  Run 13/20 complete
  Run 14/20 complete
  Run 15/20 complete
  Run 16/20 complete
  Run 17/20 complete
  Run 18/20 complete
  Run 19/20 complete
  Run 20/20 complete

RESULTS SUMMARY

With Events   
  Avg exited:      104.6 ± 3.4
  Avg steps:       381.2 ± 88.9
  Avg events/run:  32.0
  Avg latency:     99.3 steps
  P95 latency:     234.0 steps

Without Events
  Avg exited:      100.0 ± 0.0
  Avg steps:       395.7 ± 76.0
  Avg events/run:  0.0
  Avg latency:     106.7 steps
  P95 latency:     270.0 steps

HUMAN UNPREDICTABILITY IMPACT
Steps increase due to events: -3.7%
This represents the cost of human
unpredictability on queue throughput
